In [1]:
from pathlib import Path
import zipfile
import pandas as pd

In [3]:
# Project root = folder containing the "data" and "notebooks" folders
PROJECT_ROOT = Path.cwd().parent

GBIF_DIR = PROJECT_ROOT / "data" / "raw" / "gbif"

print("GBIF folder:", GBIF_DIR)
print("Exists:", GBIF_DIR.exists())

GBIF folder: d:\Biosential_India\data\raw\gbif
Exists: True


In [4]:
gbif_zips = list(GBIF_DIR.glob("*.zip"))

print("GBIF ZIP files found:", len(gbif_zips))

for f in gbif_zips:
    print(f.name)

GBIF ZIP files found: 1
0008657-260903145123482.zip


In [5]:
GBIF_ZIP = gbif_zips[0]

print("Using:", GBIF_ZIP)

Using: d:\Biosential_India\data\raw\gbif\0008657-260903145123482.zip


In [6]:
with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    print("Files inside ZIP:")
    for name in z.namelist():
        print(name)

Files inside ZIP:
0008657-260903145123482.csv


## 2. Inspect GBIF Dataset

In [7]:
with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:
        sample_df = pd.read_csv(
            f,
            sep="\t",
            nrows=10
        )

sample_df

,kingdom,kingdomkey,phylum,phylumkey,class,classkey,order,orderkey,family,familykey,...,specieskey,year,eqdcellcode,kingdomcount,phylumcount,classcount,ordercount,familycount,genuscount,occurrences
0,Animalia,N,Arthropoda,RT,Insecta,H6,Lepidoptera,B6L67,NaN,NaN,...,FLBS2,2025,E083N19DB,1497,1356,1247,817,4,2,2
1,Animalia,N,Arthropoda,RT,Insecta,H6,Lepidoptera,B6L67,NaN,NaN,...,FLBS2,2024,E085N20DA,12281,144,144,128,2,2,2
2,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,CPZZ7,2019,E071N21CA,208,208,207,14,7,3,1
3,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,CPZZN,2019,E071N21CA,208,208,207,14,7,3,1
4,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,GCHS,2019,E071N21CA,208,208,207,14,7,3,1
5,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,GCHS,2009,E071N23CD,13,13,13,4,2,2,1
6,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,CPZZN,2009,E071N23CD,13,13,13,4,2,2,1
7,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,CPZZ7,2013,E072N23DC,476,470,468,34,25,11,7
8,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,GCHS,2013,E072N23DC,476,470,468,34,25,11,2
9,Animalia,N,Chordata,CH2,Aves,V2,Pelecaniformes,3RZ,Ardeidae,6PB,...,GCHR,2013,E072N23DC,476,470,468,34,25,11,2


In [8]:
print("Shape:", sample_df.shape)
print("\nColumns:")
print(sample_df.columns.tolist())

Shape: (10, 23)

Columns:
['kingdom', 'kingdomkey', 'phylum', 'phylumkey', 'class', 'classkey', 'order', 'orderkey', 'family', 'familykey', 'genus', 'genuskey', 'species', 'specieskey', 'year', 'eqdcellcode', 'kingdomcount', 'phylumcount', 'classcount', 'ordercount', 'familycount', 'genuscount', 'occurrences']


In [9]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   kingdom       10 non-null     str  
 1   kingdomkey    10 non-null     str  
 2   phylum        10 non-null     str  
 3   phylumkey     10 non-null     str  
 4   class         10 non-null     str  
 5   classkey      10 non-null     str  
 6   order         10 non-null     str  
 7   orderkey      10 non-null     str  
 8   family        8 non-null      str  
 9   familykey     8 non-null      str  
 10  genus         8 non-null      str  
 11  genuskey      8 non-null      str  
 12  species       10 non-null     str  
 13  specieskey    10 non-null     str  
 14  year          10 non-null     int64
 15  eqdcellcode   10 non-null     str  
 16  kingdomcount  10 non-null     int64
 17  phylumcount   10 non-null     int64
 18  classcount    10 non-null     int64
 19  ordercount    10 non-null     int64
 20  fa

In [10]:
sample_df.isna().sum()

kingdom         0
kingdomkey      0
phylum          0
phylumkey       0
class           0
classkey        0
order           0
orderkey        0
family          2
familykey       2
genus           2
genuskey        2
species         0
specieskey      0
year            0
eqdcellcode     0
kingdomcount    0
phylumcount     0
classcount      0
ordercount      0
familycount     0
genuscount      0
occurrences     0
dtype: int64

## 3. Full Dataset Audit

The GBIF cube is processed in chunks to avoid loading the complete dataset
into memory. We calculate dataset-level statistics and taxonomic coverage.

In [11]:
# Columns needed for the audit
AUDIT_COLUMNS = [
    "kingdom",
    "phylum",
    "class",
    "order",
    "specieskey",
    "year",
    "eqdcellcode",
    "occurrences"
]

CHUNK_SIZE = 100_000

total_rows = 0
total_occurrences = 0

unique_species = set()
unique_cells = set()
unique_years = set()
unique_kingdoms = set()

with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            sep="\t",
            usecols=AUDIT_COLUMNS,
            chunksize=CHUNK_SIZE
        ):
            total_rows += len(chunk)

            total_occurrences += chunk["occurrences"].sum()

            unique_species.update(
                chunk["specieskey"].dropna().unique()
            )

            unique_cells.update(
                chunk["eqdcellcode"].dropna().unique()
            )

            unique_years.update(
                chunk["year"].dropna().unique()
            )

            unique_kingdoms.update(
                chunk["kingdom"].dropna().unique()
            )

print("===== GBIF DATASET AUDIT =====")
print(f"Aggregated rows       : {total_rows:,}")
print(f"Underlying occurrences: {total_occurrences:,}")
print(f"Unique species        : {len(unique_species):,}")
print(f"Unique grid cells     : {len(unique_cells):,}")
print(f"Unique years          : {len(unique_years):,}")
print(f"Unique kingdoms       : {len(unique_kingdoms):,}")
print(f"Year range            : {min(unique_years)} - {max(unique_years)}")
print(f"Kingdoms              : {sorted(unique_kingdoms)}")

===== GBIF DATASET AUDIT =====
Aggregated rows       : 3,928,428
Underlying occurrences: 62,973,959
Unique species        : 24,340
Unique grid cells     : 8,996
Unique years          : 174
Unique kingdoms       : 8
Year range            : 1820 - 2026
Kingdoms              : ['Animalia', 'Bacillati', 'Chromista', 'Fungi', 'Plantae', 'Protozoa', 'Pseudomonadati', 'Shotokuvirae']


In [12]:
# Recalculate unique species using a set of normalized species keys
all_species = set()

with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    with z.open(csv_name) as f:
        for chunk in pd.read_csv(
            f,
            sep="\t",
            usecols=["specieskey"],
            chunksize=CHUNK_SIZE
        ):
            species = (
                chunk["specieskey"]
                .dropna()
                .astype(str)
                .str.strip()
            )

            all_species.update(species.unique())

print("Verified unique species:", len(all_species))

Verified unique species: 24340


In [13]:
print("Sample species keys:", list(sorted(all_species))[:20])
print("Blank species keys:", sum(s == "" for s in all_species))

Sample species keys: ['322K8', '3237P', '323F9', '323FK', '323G4', '323G8', '325BP', '325C3', '325C9', '325CB', '325CC', '325TF', '325XH', '326B3', '326NV', '326PJ', '326PW', '326Q9', '326QL', '328BK']
Blank species keys: 0


## 4. Taxonomic Composition of Animalia

We examine the distribution of Animalia observations across taxonomic
classes and orders to determine the terrestrial scope of BioSentinel India.

In [14]:
# Aggregate Animalia observations by class
class_occurrences = {}

with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            sep="\t",
            usecols=["kingdom", "class", "occurrences"],
            chunksize=CHUNK_SIZE
        ):
            animalia = chunk[chunk["kingdom"] == "Animalia"]

            grouped = (
                animalia
                .groupby("class", dropna=False)["occurrences"]
                .sum()
            )

            for cls, count in grouped.items():
                class_occurrences[cls] = (
                    class_occurrences.get(cls, 0) + count
                )

animalia_classes = (
    pd.Series(class_occurrences, name="occurrences")
    .sort_values(ascending=False)
)

animalia_classes

Aves              61243190
Insecta             674776
Mammalia             81845
Reptilia             55292
Arachnida            41774
Amphibia             20614
Teleostei            14031
Gastropoda           12587
Malacostraca          5492
Diplopoda             2276
Anthozoa              1970
Bivalvia              1726
Hydrozoa               546
Elasmobranchii         476
Chilopoda              443
NaN                    407
Holothuroidea          395
Thecostraca            346
Echinoidea             279
Demospongiae           262
Asteroidea             233
Clitellata             123
Cephalopoda            120
Octocorallia           109
Polyplacophora          99
Ophiuroidea             94
Scyphozoa               93
Crinoidea               77
Merostomata             51
Ascidiacea              35
Polychaeta              26
Pilidiophora            23
Copepoda                14
Eurotatoria             10
Lingulata               10
Hexactinellida           7
Gymnolaemata             5
C

In [15]:
animalia_class_summary = pd.DataFrame({
    "occurrences": animalia_classes
})

animalia_class_summary["percentage"] = (
    animalia_class_summary["occurrences"]
    / animalia_class_summary["occurrences"].sum()
    * 100
)

animalia_class_summary

,occurrences,percentage
Aves,61243190,98.525277
Insecta,674776,1.085549
Mammalia,81845,0.131669
Reptilia,55292,0.088951
Arachnida,41774,0.067204
Amphibia,20614,0.033163
Teleostei,14031,0.022572
Gastropoda,12587,0.020249
Malacostraca,5492,0.008835
Diplopoda,2276,0.003662


In [16]:
print("Number of Animalia classes:", len(animalia_class_summary))
print("\nTop Animalia classes:")
print(animalia_class_summary.head(30))

Number of Animalia classes: 48

Top Animalia classes:
                occurrences  percentage
Aves               61243190   98.525277
Insecta              674776    1.085549
Mammalia              81845    0.131669
Reptilia              55292    0.088951
Arachnida             41774    0.067204
Amphibia              20614    0.033163
Teleostei             14031    0.022572
Gastropoda            12587    0.020249
Malacostraca           5492    0.008835
Diplopoda              2276    0.003662
Anthozoa               1970    0.003169
Bivalvia               1726    0.002777
Hydrozoa                546    0.000878
Elasmobranchii          476    0.000766
Chilopoda               443    0.000713
NaN                     407    0.000655
Holothuroidea           395    0.000635
Thecostraca             346    0.000557
Echinoidea              279    0.000449
Demospongiae            262    0.000421
Asteroidea              233    0.000375
Clitellata              123    0.000198
Cephalopoda             12

## 5. Order-Level Taxonomic Inspection

Order-level distributions are examined within the major terrestrial
Animalia classes to validate the terrestrial filtering strategy.

In [17]:
TERRESTRIAL_CANDIDATE_CLASSES = [
    "Aves",
    "Mammalia",
    "Reptilia",
    "Amphibia",
    "Insecta",
    "Arachnida",
    "Diplopoda",
    "Chilopoda"
]

order_occurrences = {}

with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            sep="\t",
            usecols=["kingdom", "class", "order", "occurrences"],
            chunksize=CHUNK_SIZE
        ):
            filtered = chunk[
                (chunk["kingdom"] == "Animalia") &
                (chunk["class"].isin(TERRESTRIAL_CANDIDATE_CLASSES))
            ]

            grouped = (
                filtered
                .groupby(["class", "order"], dropna=False)["occurrences"]
                .sum()
            )

            for (cls, order), count in grouped.items():
                key = (cls, order)
                order_occurrences[key] = (
                    order_occurrences.get(key, 0) + count
                )

order_summary = (
    pd.Series(order_occurrences, name="occurrences")
    .sort_values(ascending=False)
)

order_summary.head(50)

Aves       Passeriformes          29737902
           Pelecaniformes          4433184
           Charadriiformes         3562667
           Columbiformes           3257694
           Accipitriformes         2971586
           Coraciiformes           2971525
           Cuculiformes            2325424
           Piciformes              2319094
           Psittaciformes          1701255
           Anseriformes            1376767
           Galliformes             1271985
           Suliformes              1190830
           Gruiformes              1170270
           Apodiformes              681470
           Ciconiiformes            598252
           Bucerotiformes           539533
           Strigiformes             514461
Insecta    Lepidoptera              458464
Aves       Podicipediformes         251660
           Falconiformes            137734
Insecta    Odonata                  112444
Aves       Caprimulgiformes         102207
           Phoenicopteriformes       51640
Reptilia   

In [18]:
order_df = order_summary.reset_index()
order_df.columns = ["class", "order", "occurrences"]

order_df["percentage_within_class"] = (
    order_df["occurrences"]
    / order_df.groupby("class")["occurrences"].transform("sum")
    * 100
)

order_df.head(50)

,class,order,occurrences,percentage_within_class
0,Aves,Passeriformes,29737902,48.557075
1,Aves,Pelecaniformes,4433184,7.238656
2,Aves,Charadriiformes,3562667,5.817246
3,Aves,Columbiformes,3257694,5.319275
4,Aves,Accipitriformes,2971586,4.852108
5,Aves,Coraciiformes,2971525,4.852009
6,Aves,Cuculiformes,2325424,3.797033
7,Aves,Piciformes,2319094,3.786697
8,Aves,Psittaciformes,1701255,2.777868
9,Aves,Anseriformes,1376767,2.248033


In [19]:
# Candidate terrestrial classes
candidate_class_totals = (
    order_df.groupby("class")["occurrences"]
    .sum()
    .sort_values(ascending=False)
)

print(candidate_class_totals)

class
Aves         61243190
Insecta        674776
Mammalia        81845
Reptilia        55292
Arachnida       41774
Amphibia        20614
Diplopoda        2276
Chilopoda         443
Name: occurrences, dtype: int64


In [20]:
orders_to_check = [
    "Cetacea",
    "Sirenia",
    "Chiroptera",
    "Testudines",
    "Crocodylia",
    "Anseriformes",
    "Pelecaniformes",
    "Suliformes",
    "Procellariiformes",
    "Podicipediformes",
    "Phoenicopteriformes",
    "Charadriiformes",
    "Odonata"
]

aquatic_check = order_df[
    order_df["order"].isin(orders_to_check)
].sort_values("occurrences", ascending=False)

aquatic_check

,class,order,occurrences,percentage_within_class
1,Aves,Pelecaniformes,4433184,7.238656
2,Aves,Charadriiformes,3562667,5.817246
9,Aves,Anseriformes,1376767,2.248033
11,Aves,Suliformes,1190830,1.944428
18,Aves,Podicipediformes,251660,0.410919
20,Insecta,Odonata,112444,16.663900
22,Aves,Phoenicopteriformes,51640,0.084320
35,Aves,Procellariiformes,10369,0.016931
38,Reptilia,Testudines,4144,7.494755
42,Mammalia,Chiroptera,2675,3.268373


## 6. Terrestrial Biodiversity Scope

For the initial BioSentinel analysis, terrestrial-associated Animalia
classes are retained. Clearly aquatic taxonomic classes are excluded.
Mixed ecological classes are retained and can be refined at species level
where required.

In [21]:
TERRESTRIAL_CLASSES = {
    "Aves",
    "Insecta",
    "Mammalia",
    "Reptilia",
    "Arachnida",
    "Amphibia",
    "Diplopoda",
    "Chilopoda"
}

print("Terrestrial classes:")
for cls in sorted(TERRESTRIAL_CLASSES):
    print("-", cls)

Terrestrial classes:
- Amphibia
- Arachnida
- Aves
- Chilopoda
- Diplopoda
- Insecta
- Mammalia
- Reptilia


In [22]:
terrestrial_occurrences = 0
terrestrial_rows = 0

with zipfile.ZipFile(GBIF_ZIP, "r") as z:
    with z.open(csv_name) as f:
        for chunk in pd.read_csv(
            f,
            sep="\t",
            usecols=["kingdom", "class", "occurrences"],
            chunksize=CHUNK_SIZE
        ):
            mask = (
                (chunk["kingdom"] == "Animalia") &
                (chunk["class"].isin(TERRESTRIAL_CLASSES))
            )

            terrestrial_rows += mask.sum()
            terrestrial_occurrences += chunk.loc[mask, "occurrences"].sum()

print("Terrestrial aggregated rows :", f"{terrestrial_rows:,}")
print("Terrestrial occurrences     :", f"{terrestrial_occurrences:,}")
print(
    "Share of all occurrences    :",
    f"{terrestrial_occurrences / total_occurrences * 100:.2f}%"
)

Terrestrial aggregated rows : 3,664,155
Terrestrial occurrences     : 62,120,210
Share of all occurrences    : 98.64%


## 7. Create Terrestrial Analytical Dataset

The filtered terrestrial observations are written to a Parquet file for
efficient downstream biodiversity, spatial-temporal, and anomaly analysis.

In [24]:
import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "gbif_terrestrial_species_year_grid.parquet"
)

writer = None
written_rows = 0

try:
    with zipfile.ZipFile(GBIF_ZIP, "r") as z:
        with z.open(csv_name) as f:

            for chunk in pd.read_csv(
                f,
                sep="\t",
                usecols=OUTPUT_COLUMNS,
                chunksize=CHUNK_SIZE
            ):

                mask = (
                    (chunk["kingdom"] == "Animalia") &
                    (chunk["class"].isin(TERRESTRIAL_CLASSES))
                )

                terrestrial_chunk = chunk.loc[mask].copy()

                if terrestrial_chunk.empty:
                    continue

                table = pa.Table.from_pandas(
                    terrestrial_chunk,
                    preserve_index=False
                )

                if writer is None:
                    writer = pq.ParquetWriter(
                        OUTPUT_PATH,
                        table.schema,
                        compression="snappy"
                    )

                writer.write_table(table)
                written_rows += len(terrestrial_chunk)

finally:
    if writer is not None:
        writer.close()

print(f"Created: {OUTPUT_PATH}")
print(f"Rows written: {written_rows:,}")
print(f"File size: {OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

Created: d:\Biosential_India\data\processed\gbif_terrestrial_species_year_grid.parquet
Rows written: 3,664,155
File size: 27.66 MB


## 8. Verify Processed Dataset

In [25]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(OUTPUT_PATH)

print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)
print("\nSchema:")
print(parquet_file.schema)

Rows: 3664155
Columns: 10

Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 kingdom (String);
  optional binary field_id=-1 class (String);
  optional binary field_id=-1 order (String);
  optional binary field_id=-1 family (String);
  optional binary field_id=-1 genus (String);
  optional binary field_id=-1 species (String);
  optional binary field_id=-1 specieskey (String);
  optional int64 field_id=-1 year;
  optional binary field_id=-1 eqdcellcode (String);
  optional int64 field_id=-1 occurrences;
}



In [26]:
print("File size:",
      f"{OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB")

File size: 27.66 MB


In [29]:
import pyarrow.parquet as pq

table = pq.read_table(
    OUTPUT_PATH,
    columns=["kingdom", "class", "year", "occurrences"]
)

print("Rows:", table.num_rows)
print("Columns:", table.num_columns)

print("\nClasses:")
print(
    sorted(
        set(
            table.column("class")
            .drop_null()
            .to_pylist()
        )
    )
)

years = table.column("year").to_pylist()
occurrences = table.column("occurrences").to_pylist()

print("\nYear range:")
print(min(years), "to", max(years))

print("\nTotal occurrences:")
print(f"{sum(occurrences):,}")

Rows: 3664155
Columns: 4

Classes:
['Amphibia', 'Arachnida', 'Aves', 'Chilopoda', 'Diplopoda', 'Insecta', 'Mammalia', 'Reptilia']

Year range:
1820 to 2026

Total occurrences:
62,120,210


## 9. Data Quality Assessment

We assess missing taxonomy, temporal coverage, spatial coverage, and
potential duplicate Species × Year × Grid Cell combinations before
generating biodiversity indicators.

In [30]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(OUTPUT_PATH)

print("Rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

Rows: 3664155
Row groups: 40


In [31]:
columns = [
    "kingdom",
    "class",
    "order",
    "family",
    "genus",
    "species",
    "specieskey",
    "year",
    "eqdcellcode",
    "occurrences"
]

missing_counts = {col: 0 for col in columns}
total_rows_checked = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=columns
):
    df = batch.to_pandas()

    total_rows_checked += len(df)

    for col in columns:
        missing_counts[col] += df[col].isna().sum()

missing_df = pd.DataFrame.from_dict(
    missing_counts,
    orient="index",
    columns=["missing_count"]
)

missing_df["missing_percentage"] = (
    missing_df["missing_count"]
    / total_rows_checked
    * 100
)

missing_df

,missing_count,missing_percentage
kingdom,0,0.000000
class,0,0.000000
order,1110,0.030293
family,1109,0.030266
genus,1424,0.038863
species,0,0.000000
specieskey,0,0.000000
year,0,0.000000
eqdcellcode,0,0.000000
occurrences,0,0.000000


## 10. Temporal Coverage Assessment

We examine the distribution of observations across years to identify
historical sparsity and incomplete recent-year coverage.

In [32]:
year_occurrences = {}
year_rows = {}

pf = pq.ParquetFile(OUTPUT_PATH)

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["year", "occurrences"]
):
    df = batch.to_pandas()

    grouped_occ = df.groupby("year")["occurrences"].sum()
    grouped_rows = df.groupby("year").size()

    for year, value in grouped_occ.items():
        year_occurrences[year] = (
            year_occurrences.get(year, 0) + value
        )

    for year, value in grouped_rows.items():
        year_rows[year] = (
            year_rows.get(year, 0) + value
        )

temporal_df = pd.DataFrame({
    "year": sorted(year_occurrences.keys()),
    "aggregated_rows": [
        year_rows[y] for y in sorted(year_occurrences.keys())
    ],
    "occurrences": [
        year_occurrences[y] for y in sorted(year_occurrences.keys())
    ]
})

temporal_df["percentage"] = (
    temporal_df["occurrences"]
    / temporal_df["occurrences"].sum()
    * 100
)

temporal_df.head(20)

,year,aggregated_rows,occurrences,percentage
0,1820,1,1,0.000002
1,1837,1,1,0.000002
2,1840,2,2,0.000003
3,1842,1,1,0.000002
4,1844,1,1,0.000002
5,1846,2,2,0.000003
6,1847,1,1,0.000002
7,1849,10,12,0.000019
8,1853,3,3,0.000005
9,1858,1,1,0.000002


In [33]:
temporal_df[
    temporal_df["year"] >= 2014
]

,year,aggregated_rows,occurrences,percentage
161,2014,112572,755610,1.216367
162,2015,142919,1585704,2.552638
163,2016,171905,2505770,4.033744
164,2017,204644,3385548,5.449994
165,2018,229205,3934826,6.334212
166,2019,259279,4538443,7.305904
167,2020,272165,5717371,9.203721
168,2021,319664,7180361,11.558816
169,2022,354552,8396943,13.517248
170,2023,394653,10025664,16.139134


In [34]:
print("Lowest observation years:")
print(
    temporal_df
    .sort_values("occurrences")
    .head(10)
)

print("\nHighest observation years:")
print(
    temporal_df
    .sort_values("occurrences", ascending=False)
    .head(10)
)

Lowest observation years:
    year  aggregated_rows  occurrences  percentage
0   1820                1            1    0.000002
1   1837                1            1    0.000002
3   1842                1            1    0.000002
4   1844                1            1    0.000002
6   1847                1            1    0.000002
13  1864                1            1    0.000002
12  1863                1            1    0.000002
9   1858                1            1    0.000002
22  1873                1            1    0.000002
44  1897                1            1    0.000002

Highest observation years:
     year  aggregated_rows  occurrences  percentage
171  2024           420515     11811996   19.014739
170  2023           394653     10025664   16.139134
169  2022           354552      8396943   13.517248
168  2021           319664      7180361   11.558816
167  2020           272165      5717371    9.203721
166  2019           259279      4538443    7.305904
165  2018           2

## 11. Spatial Coverage Assessment

We evaluate the distribution of biodiversity observations across GBIF
Equal-area grid cells to identify geographic observation-density patterns
and areas with limited sampling coverage.

In [35]:
spatial_occurrences = {}
spatial_rows = {}

pf = pq.ParquetFile(OUTPUT_PATH)

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["eqdcellcode", "occurrences"]
):
    df = batch.to_pandas()

    grouped_occ = df.groupby("eqdcellcode")["occurrences"].sum()
    grouped_rows = df.groupby("eqdcellcode").size()

    for cell, value in grouped_occ.items():
        spatial_occurrences[cell] = (
            spatial_occurrences.get(cell, 0) + value
        )

    for cell, value in grouped_rows.items():
        spatial_rows[cell] = (
            spatial_rows.get(cell, 0) + value
        )

spatial_df = pd.DataFrame({
    "eqdcellcode": sorted(spatial_occurrences.keys()),
    "aggregated_rows": [
        spatial_rows[c] for c in sorted(spatial_occurrences.keys())
    ],
    "occurrences": [
        spatial_occurrences[c] for c in sorted(spatial_occurrences.keys())
    ]
})

spatial_df["percentage"] = (
    spatial_df["occurrences"]
    / spatial_df["occurrences"].sum()
    * 100
)

print("Grid cells:", len(spatial_df))
print("\nTop 20 cells by observations:")
spatial_df.sort_values(
    "occurrences",
    ascending=False
).head(20)

Grid cells: 8434

Top 20 cells by observations:


,eqdcellcode,aggregated_rows,occurrences,percentage
3516,E077N12BA,12171,2180802,3.510616
3542,E077N13DC,9136,1475070,2.374541
3031,E076N12DA,9925,826617,1.330673
1482,E072N19DD,11585,753953,1.213700
1801,E073N18BD,8598,748825,1.205445
2972,E076N09AB,6076,715129,1.151202
4916,E080N12AA,8179,691154,1.112607
3764,E077N27DC,13895,653052,1.051271
5078,E080N22DC,5252,608023,0.978785
3018,E076N11DD,5481,598888,0.964079


In [36]:
print("\nSpatial occurrence statistics:")
print(
    spatial_df["occurrences"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


Spatial occurrence statistics:
count    8.434000e+03
mean     7.365451e+03
std      4.742117e+04
min      1.000000e+00
25%      1.000000e+00
50%      3.900000e+01
75%      1.049000e+03
90%      8.224000e+03
95%      2.748205e+04
99%      1.505714e+05
max      2.180802e+06
Name: occurrences, dtype: float64


In [37]:
print("\nLowest-observation cells:")
print(
    spatial_df
    .sort_values("occurrences")
    .head(20)
)


Lowest-observation cells:
   eqdcellcode  aggregated_rows  occurrences  percentage
48   E050N26CA                1            1    0.000002
1    E003N67DD                1            1    0.000002
18   E023N04DD                1            1    0.000002
19   E023N33DC                1            1    0.000002
20   E024N11CB                1            1    0.000002
21   E025N59CD                1            1    0.000002
22   E025S31AA                1            1    0.000002
23   E026S60DA                1            1    0.000002
24   E029N11DD                1            1    0.000002
25   E029N34BD                1            1    0.000002
26   E032S31DC                1            1    0.000002
43   E048S66DB                1            1    0.000002
44   E049N23DD                1            1    0.000002
45   E049N36DD                1            1    0.000002
46   E049S20DA                1            1    0.000002
63   E054N05AA                1            1    0.000002
38  

## 12. Spatial Sampling Coverage

For each grid cell, observation volume, species representation, and temporal
coverage are calculated to characterize sampling effort.

In [38]:
cell_stats = {}

pf = pq.ParquetFile(OUTPUT_PATH)

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=[
        "eqdcellcode",
        "specieskey",
        "year",
        "occurrences"
    ]
):
    df = batch.to_pandas()

    for cell, group in df.groupby("eqdcellcode"):

        if cell not in cell_stats:
            cell_stats[cell] = {
                "occurrences": 0,
                "species": set(),
                "years": set()
            }

        cell_stats[cell]["occurrences"] += group["occurrences"].sum()

        cell_stats[cell]["species"].update(
            group["specieskey"].dropna().unique()
        )

        cell_stats[cell]["years"].update(
            group["year"].dropna().unique()
        )

In [39]:
spatial_coverage_df = pd.DataFrame([
    {
        "eqdcellcode": cell,
        "occurrences": stats["occurrences"],
        "species_count": len(stats["species"]),
        "year_count": len(stats["years"])
    }
    for cell, stats in cell_stats.items()
])

spatial_coverage_df = spatial_coverage_df.sort_values(
    "occurrences",
    ascending=False
).reset_index(drop=True)

spatial_coverage_df.head(20)

,eqdcellcode,occurrences,species_count,year_count
0,E077N12BA,2180802,1806,51
1,E077N13DC,1475070,1481,47
2,E076N12DA,826617,1170,55
3,E072N19DD,753953,1714,64
4,E073N18BD,748825,1325,50
5,E076N09AB,715129,1141,43
6,E080N12AA,691154,1311,43
7,E077N27DC,653052,621,55
8,E080N22DC,608023,708,32
9,E076N11DD,598888,880,40


In [40]:
print("===== SPATIAL COVERAGE =====")

print("\nSpecies count per cell:")
print(
    spatial_coverage_df["species_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nYears represented per cell:")
print(
    spatial_coverage_df["year_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

===== SPATIAL COVERAGE =====

Species count per cell:
count    8434.000000
mean      106.498340
std       171.155621
min         1.000000
25%         1.000000
50%        27.000000
75%       159.000000
90%       291.700000
95%       410.350000
99%       770.680000
max      2326.000000
Name: species_count, dtype: float64

Years represented per cell:
count    8434.000000
mean        7.805786
std         8.851395
min         1.000000
25%         1.000000
50%         4.000000
75%        12.000000
90%        20.000000
95%        25.000000
99%        39.000000
max        66.000000
Name: year_count, dtype: float64


In [41]:
low_coverage_cells = spatial_coverage_df[
    (spatial_coverage_df["occurrences"] <= 10) |
    (spatial_coverage_df["species_count"] <= 2) |
    (spatial_coverage_df["year_count"] <= 2)
].copy()

print("Potentially data-limited cells:", len(low_coverage_cells))

low_coverage_cells.head(30)

Potentially data-limited cells: 3980


,eqdcellcode,occurrences,species_count,year_count
2305,E077N19AB,823,107,2
2789,E078N26DC,444,80,2
2954,E079N25AC,360,89,2
3115,E085N21CB,296,74,2
3279,E078N28DA,234,62,2
3350,E077N22DC,211,85,2
3465,E088N23DB,177,40,2
3476,E077N24DB,175,83,1
3607,E079N25AD,143,31,1
3614,E077N18CB,142,68,1


## 13. Analytical Group Uniqueness

The processed dataset is expected to contain one aggregated record for each
Species × Year × Grid Cell combination.

In [42]:
import pyarrow.parquet as pq
import pandas as pd

pf = pq.ParquetFile(OUTPUT_PATH)

key_parts = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["specieskey", "year", "eqdcellcode"]
):
    batch_df = batch.to_pandas()
    key_parts.append(batch_df)

keys_df = pd.concat(key_parts, ignore_index=True)

print("Rows loaded:", len(keys_df))

Rows loaded: 3664155


In [43]:
duplicate_mask = keys_df.duplicated(
    subset=["specieskey", "year", "eqdcellcode"],
    keep=False
)

duplicate_rows = keys_df[duplicate_mask]

print("Duplicate rows:", len(duplicate_rows))
print(
    "Duplicate analytical groups:",
    duplicate_rows[
        ["specieskey", "year", "eqdcellcode"]
    ].drop_duplicates().shape[0]
)

Duplicate rows: 0
Duplicate analytical groups: 0


In [44]:
if len(duplicate_rows) > 0:
    print(
        duplicate_rows
        .sort_values(["specieskey", "year", "eqdcellcode"])
        .head(30)
    )
else:
    print("No duplicate Species × Year × Grid Cell combinations found.")

No duplicate Species × Year × Grid Cell combinations found.


## Dataset V1 — Quality Control Checkpoint

The processed GBIF terrestrial dataset contains 3,664,155 aggregated
Species × Year × Grid Cell records representing 62,120,210 observations
across 8,434 occupied grid cells.

Critical analytical fields (`specieskey`, `year`, and `eqdcellcode`) have
no missing values, and no duplicate Species × Year × Grid Cell combinations
were detected.

Higher-level taxonomic fields contain a small amount of missing data and
are retained because they do not prevent species-level spatial-temporal
analysis.

Temporal observation coverage is highly uneven, and spatial sampling is
also strongly heterogeneous. These limitations will be explicitly
incorporated into downstream biodiversity indicators and priority scoring.